<a href="https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from IPython.display import display

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{safe_token}'
    );
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/data_0.parquet"
)

print("Warehouse connection ready.")
print("Lane: Refresh / Content Opportunity Scoring")
print("Development month: March 2026")
print("June 2026 remains sealed.")

Warehouse connection ready.
Lane: Refresh / Content Opportunity Scoring
Development month: March 2026
June 2026 remains sealed.


## 1. My rule and its reason codes

### Signal verdicts

**Signal 1 — CTR relative to position: OPPOSITE**

Among content with at least 20 recent impressions and at least one recent click, future decline increased as CTR became stronger relative to the median CTR for its position bucket. The future-decline rate rose from 38.30% in the far-below-expected bucket to 64.59% in the above-expected bucket.

This is opposite to my original idea that unusually low CTR would identify content most likely to decline next. Therefore, I will not use low CTR as a future-decline risk signal.

**Signal 2 — Recent click direction: OPPOSITE**

Content whose clicks were already falling had a 17.94% future-decline rate, while content whose clicks had recently increased had a 65.69% future-decline rate.

This is also opposite to my original assumption. The result may reflect short-term mean reversion: a strong recent week is harder to exceed in the following week.

### Baseline rule

Because both checks contradicted my original rule, I will not force the original low-CTR + falling-clicks logic.

My baseline rule is:

**Review first the content items whose recent clicks increased and whose CTR is above expected for their current search-position bucket. Among those items, higher recent search exposure receives higher priority.**

This is a decision-support rule, not a claim that the content itself caused the future decline.

**Reason code:** `RECENT_SURGE_ABOVE_EXPECTED_CTR`

**Action label:** `REVIEW`

In [2]:
# Section 1 — Build a decision-time frame and audit two signals.
# Future data is used ONLY as an evaluation outcome,
# never as an input to the eventual baseline score.

signal_frame = con.sql(f"""
    WITH daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_sum_position
        FROM read_parquet('{MARCH_DAILY}')
        WHERE gsc_data_available IS TRUE
    ),

    windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT report_date) AS available_days,

            -- Previous 7 days: March 11–17
            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-11'
                                          AND DATE '2026-03-17'
                     THEN COALESCE(gsc_impressions, 0)
                     ELSE 0 END
            ) AS previous_impressions_7d,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-11'
                                          AND DATE '2026-03-17'
                     THEN COALESCE(gsc_clicks, 0)
                     ELSE 0 END
            ) AS previous_clicks_7d,

            -- Recent 7 days: March 18–24
            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-18'
                                          AND DATE '2026-03-24'
                     THEN COALESCE(gsc_impressions, 0)
                     ELSE 0 END
            ) AS recent_impressions_7d,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-18'
                                          AND DATE '2026-03-24'
                     THEN COALESCE(gsc_clicks, 0)
                     ELSE 0 END
            ) AS recent_clicks_7d,

            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-18'
                                          AND DATE '2026-03-24'
                     THEN COALESCE(gsc_sum_position, 0)
                     ELSE 0 END
            ) AS recent_sum_position_7d,

            -- Future outcome: March 25–31
            SUM(
                CASE WHEN report_date BETWEEN DATE '2026-03-25'
                                          AND DATE '2026-03-31'
                     THEN COALESCE(gsc_clicks, 0)
                     ELSE 0 END
            ) AS future_clicks_7d

        FROM daily
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT *
    FROM windows
    WHERE available_days = 31
      AND previous_impressions_7d > 0
      AND recent_impressions_7d > 0
""").df()


# -------------------------------------------------------
# Decision-time derived signals
# -------------------------------------------------------

signal_frame["recent_ctr"] = (
    signal_frame["recent_clicks_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["recent_avg_position"] = (
    signal_frame["recent_sum_position_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["pre_decision_click_change_7d"] = (
    signal_frame["recent_clicks_7d"]
    - signal_frame["previous_clicks_7d"]
)

# Evaluation outcome ONLY — never a baseline input
signal_frame["future_click_decline"] = (
    signal_frame["future_clicks_7d"]
    < signal_frame["recent_clicks_7d"]
).astype(int)


# =======================================================
# SIGNAL 1 — CTR relative to position
# =======================================================

signal1 = signal_frame[
    signal_frame["recent_impressions_7d"] >= 20
].copy()

signal1["position_bucket"] = pd.cut(
    signal1["recent_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "20+"
    ],
    include_lowest=True
)

# Expected CTR = median CTR among content in a similar position band
signal1["expected_ctr_for_position"] = (
    signal1
    .groupby("position_bucket", observed=True)["recent_ctr"]
    .transform("median")
)

signal1["ctr_vs_expected"] = (
    signal1["recent_ctr"]
    / signal1["expected_ctr_for_position"].replace(0, np.nan)
)

signal1["ctr_bucket"] = pd.cut(
    signal1["ctr_vs_expected"],
    bins=[-np.inf, 0.50, 0.80, 1.20, np.inf],
    labels=[
        "far_below_expected",
        "below_expected",
        "near_expected",
        "above_expected"
    ]
)

signal1_table = (
    signal1
    .dropna(subset=["ctr_bucket"])
    .groupby("ctr_bucket", observed=True)
    .agg(
        n=("future_click_decline", "size"),
        future_decline_rate=("future_click_decline", "mean"),
        median_recent_ctr=("recent_ctr", "median"),
        median_ctr_vs_expected=("ctr_vs_expected", "median")
    )
    .reset_index()
)

signal1_table["future_decline_rate"] = (
    signal1_table["future_decline_rate"] * 100
).round(2)

signal1_table["median_recent_ctr"] = (
    signal1_table["median_recent_ctr"] * 100
).round(3)

signal1_table["median_ctr_vs_expected"] = (
    signal1_table["median_ctr_vs_expected"]
).round(3)

print("SIGNAL 1 — CTR relative to position")
print(f"n = {len(signal1):,}")
display(signal1_table)


# =======================================================
# SIGNAL 2 — Pre-decision click direction
# =======================================================

signal2 = signal_frame.copy()

signal2["click_direction"] = np.select(
    [
        signal2["pre_decision_click_change_7d"] < 0,
        signal2["pre_decision_click_change_7d"] == 0,
        signal2["pre_decision_click_change_7d"] > 0
    ],
    [
        "down",
        "flat",
        "up"
    ],
    default="unknown"
)

signal2_table = (
    signal2
    .groupby("click_direction")
    .agg(
        n=("future_click_decline", "size"),
        future_decline_rate=("future_click_decline", "mean"),
        median_click_change=(
            "pre_decision_click_change_7d",
            "median"
        )
    )
    .reset_index()
)

signal2_table["future_decline_rate"] = (
    signal2_table["future_decline_rate"] * 100
).round(2)

print("\nSIGNAL 2 — Recent click direction")
print(f"n = {len(signal2):,}")
display(signal2_table)

print(
    "\nImportant: future_click_decline is used only "
    "to judge the signals. It will not be an input "
    "to the baseline score."
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — CTR relative to position
n = 61,597


,ctr_bucket,n,future_decline_rate,median_recent_ctr,median_ctr_vs_expected
0,far_below_expected,17585,0.26,0.000,0.000
1,below_expected,429,27.27,0.052,0.665
2,near_expected,741,30.63,0.078,1.017
3,above_expected,17952,54.55,0.431,5.579



SIGNAL 2 — Recent click direction
n = 61,796


,click_direction,n,future_decline_rate,median_click_change
0,down,16417,17.94,-1.0
1,flat,30601,7.58,0.0
2,up,14778,65.69,1.0



Important: future_click_decline is used only to judge the signals. It will not be an input to the baseline score.


In [3]:
# Re-check Signal 1 after removing the structural zero-click problem.
# A page with zero recent clicks cannot have a "future click decline"
# under the current label definition.

signal1_clean = signal_frame[
    (signal_frame["recent_impressions_7d"] >= 20)
    & (signal_frame["recent_clicks_7d"] > 0)
].copy()

signal1_clean["recent_ctr"] = (
    signal1_clean["recent_clicks_7d"]
    / signal1_clean["recent_impressions_7d"]
)

signal1_clean["recent_avg_position"] = (
    signal1_clean["recent_sum_position_7d"]
    / signal1_clean["recent_impressions_7d"]
)

signal1_clean["position_bucket"] = pd.cut(
    signal1_clean["recent_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

signal1_clean["expected_ctr_for_position"] = (
    signal1_clean
    .groupby("position_bucket", observed=True)["recent_ctr"]
    .transform("median")
)

signal1_clean["ctr_vs_expected"] = (
    signal1_clean["recent_ctr"]
    / signal1_clean["expected_ctr_for_position"].replace(0, np.nan)
)

signal1_clean["ctr_bucket"] = pd.cut(
    signal1_clean["ctr_vs_expected"],
    bins=[-np.inf, 0.50, 0.80, 1.20, np.inf],
    labels=[
        "far_below_expected",
        "below_expected",
        "near_expected",
        "above_expected"
    ]
)

signal1_clean_table = (
    signal1_clean
    .dropna(subset=["ctr_bucket"])
    .groupby("ctr_bucket", observed=True)
    .agg(
        n=("future_click_decline", "size"),
        future_decline_rate=("future_click_decline", "mean"),
        median_recent_ctr=("recent_ctr", "median"),
        median_ctr_vs_expected=("ctr_vs_expected", "median")
    )
    .reset_index()
)

signal1_clean_table["future_decline_rate"] = (
    signal1_clean_table["future_decline_rate"] * 100
).round(2)

signal1_clean_table["median_recent_ctr"] = (
    signal1_clean_table["median_recent_ctr"] * 100
).round(3)

signal1_clean_table["median_ctr_vs_expected"] = (
    signal1_clean_table["median_ctr_vs_expected"]
).round(3)

print("SIGNAL 1 — CTR relative to position, positive recent clicks only")
print(f"n = {len(signal1_clean):,}")
display(signal1_clean_table)

SIGNAL 1 — CTR relative to position, positive recent clicks only
n = 27,186


,ctr_bucket,n,future_decline_rate,median_recent_ctr,median_ctr_vs_expected
0,far_below_expected,6603,38.30,0.118,0.319
1,below_expected,4584,51.29,0.253,0.643
2,near_expected,4388,58.87,0.388,0.981
3,above_expected,11611,64.59,0.756,1.953


## 2. Build the ranked queue (writes the CSV)

### Ranked queue

The rule uses only information available by March 24, 2026.

A content item becomes eligible when:

1. its clicks increased from March 11–17 to March 18–24; and
2. its recent CTR is more than 1.20 times the median CTR for its search-position bucket.

For eligible items, I use recent impressions as the ranking score so that higher-exposure items are reviewed first.

The future outcome is used only to evaluate the baseline. It is not part of the score.
### Baseline result

The baseline selected 8,039 of 27,186 eligible rows.

Its Precision@20 was 55.00%, while the overall future-decline base rate was 55.04%. Therefore, this rule did not provide measurable lift at the top 20.

I will keep it as an honest, transparent baseline rather than tune thresholds after seeing the outcome. The Week-5 model should improve on this result.

In [4]:
# Section 2 — Encode ONE transparent baseline rule
# and build the ranked action queue.

import os
import json

baseline = signal1_clean.copy()

# -------------------------------------------------------
# ONE RULE
# -------------------------------------------------------

baseline["recent_click_increase"] = (
    baseline["pre_decision_click_change_7d"] > 0
)

baseline["above_expected_ctr"] = (
    baseline["ctr_vs_expected"] > 1.20
)

baseline["rule_match"] = (
    baseline["recent_click_increase"]
    & baseline["above_expected_ctr"]
)

# Simple transparent score:
# only matching rows receive a score;
# more recent impressions = higher review priority.
baseline["baseline_action_score"] = np.where(
    baseline["rule_match"],
    baseline["recent_impressions_7d"],
    0
)

baseline["reason_code"] = np.where(
    baseline["rule_match"],
    "RECENT_SURGE_ABOVE_EXPECTED_CTR",
    "NOT_SELECTED"
)

baseline["action_label"] = np.where(
    baseline["rule_match"],
    "REVIEW",
    "NO_ACTION"
)

# Rank matching rows first.
queue = (
    baseline[
        baseline["rule_match"]
    ]
    .sort_values(
        "baseline_action_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)


# -------------------------------------------------------
# Honest baseline evaluation
# -------------------------------------------------------

base_rate = baseline["future_click_decline"].mean()

TOP_K = 20

precision_at_20 = (
    queue.head(TOP_K)["future_click_decline"].mean()
    if len(queue) >= TOP_K
    else np.nan
)

print(f"Rows evaluated: {len(baseline):,}")
print(f"Rows selected by rule: {len(queue):,}")
print(f"Future-decline base rate: {base_rate:.4f}")
print(f"Precision@20: {precision_at_20:.4f}")


# -------------------------------------------------------
# Write the required local CSV.
# Future label is NOT written into the action queue.
# -------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue_output = queue[
    [
        "client_hash_id",
        "content_hash_id",
        "rank",
        "baseline_action_score",
        "reason_code",
        "action_label",
        "recent_impressions_7d",
        "recent_clicks_7d",
        "recent_avg_position",
        "recent_ctr",
        "ctr_vs_expected",
        "pre_decision_click_change_7d",
    ]
].copy()

queue_output.to_csv(
    output_path,
    index=False
)

print(f"\nQueue written to: {output_path}")


# -------------------------------------------------------
# Save a small metrics receipt — safe to commit.
# -------------------------------------------------------

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "development_month": "2026-03",
    "rule": (
        "recent clicks increased AND "
        "CTR > 1.20x expected for position"
    ),
    "reason_code": "RECENT_SURGE_ABOVE_EXPECTED_CTR",
    "action_label": "REVIEW",
    "rows_evaluated": int(len(baseline)),
    "rows_selected": int(len(queue)),
    "base_rate": round(float(base_rate), 4),
    "precision_at_20": (
        round(float(precision_at_20), 4)
        if not np.isnan(precision_at_20)
        else None
    ),
}

metrics_path = os.path.join(
    output_dir,
    "w04_baseline_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics receipt written to: {metrics_path}")


# -------------------------------------------------------
# Safe notebook preview — no hash IDs displayed
# -------------------------------------------------------

display_columns = [
    "rank",
    "baseline_action_score",
    "action_label",
    "reason_code",
    "recent_impressions_7d",
    "recent_clicks_7d",
    "recent_avg_position",
    "recent_ctr",
    "ctr_vs_expected",
    "pre_decision_click_change_7d",
]

print("\nTop 20 preview:")
display(queue[display_columns].head(20))


Rows evaluated: 27,186
Rows selected by rule: 8,039
Future-decline base rate: 0.5504
Precision@20: 0.5500

Queue written to: work/outputs/baseline_action_score.csv
Metrics receipt written to: work/outputs/w04_baseline_metrics.json

Top 20 preview:


,rank,baseline_action_score,action_label,reason_code,recent_impressions_7d,recent_clicks_7d,recent_avg_position,recent_ctr,ctr_vs_expected,pre_decision_click_change_7d
0,1,52735.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,52735.0,710.0,4.240429,0.013464,3.365886,219.0
1,2,41251.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,41251.0,624.0,3.139197,0.015127,3.781727,204.0
2,3,34792.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,34792.0,227.0,11.970740,0.006524,1.409289,158.0
3,4,33994.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,33994.0,167.0,3.765194,0.004913,1.228158,98.0
4,5,30538.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,30538.0,201.0,5.253291,0.006582,1.645491,32.0
5,6,29010.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,29010.0,280.0,3.505102,0.009652,2.412961,186.0
6,7,28999.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,28999.0,170.0,3.368358,0.005862,1.465568,66.0
7,8,24453.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,24453.0,215.0,2.784975,0.008792,2.386251,121.0
8,9,22465.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,22465.0,275.0,3.374894,0.012241,3.060316,137.0
9,10,22407.0,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,22407.0,177.0,3.046280,0.007899,1.974829,75.0


## 3. Top-20 review

### Top-20 skeptical review

All twenty rows receive the action **REVIEW**, not an automatic refresh recommendation.

The reason code is `RECENT_SURGE_ABOVE_EXPECTED_CTR`: each item had increasing recent clicks and CTR above the expected level for its position bucket.

Because Precision@20 (55.00%) is effectively equal to the base rate (55.04%), I treat these recommendations cautiously. High exposure makes the items worth looking at, but the rule alone does not establish that a refresh is needed.

1. **Rank 1 — REVIEW.** Very high exposure (52,735 impressions), clicks increased by 219, and CTR was 3.37× expected. **Confidence: medium-high.** Wrong if the surge reflects healthy sustained demand rather than a temporary peak.

2. **Rank 2 — REVIEW.** 41,251 impressions, +204 clicks, and CTR 3.78× expected. **Confidence: medium-high.** Wrong if strong CTR reflects stable branded or navigational demand that does not need intervention.

3. **Rank 3 — REVIEW.** 34,792 impressions and +158 clicks, but average position was around 12 and CTR only 1.41× expected. **Confidence: medium.** Wrong if the recent increase represents genuine ranking improvement that continues.

4. **Rank 4 — REVIEW.** High exposure and +98 clicks, but CTR is only 1.23× expected, barely above my rule threshold. **Confidence: low-medium.** Wrong if this is simply normal variation around the threshold.

5. **Rank 5 — REVIEW.** 30,538 impressions, +32 clicks, and CTR 1.65× expected. **Confidence: medium.** Wrong if the modest click increase is noise rather than a meaningful change.

6. **Rank 6 — REVIEW.** 29,010 impressions, +186 clicks, and CTR 2.41× expected. **Confidence: medium-high.** Wrong if the surge is driven by a continuing increase in search demand.

7. **Rank 7 — REVIEW.** 28,999 impressions, +66 clicks, and CTR 1.47× expected. **Confidence: medium.** Wrong if current performance is stable and the next-week movement is normal fluctuation.

8. **Rank 8 — REVIEW.** 24,453 impressions, +121 clicks, and CTR 2.39× expected. **Confidence: medium-high.** Wrong if recent growth is durable rather than a short-lived spike.

9. **Rank 9 — REVIEW.** 22,465 impressions, +137 clicks, and CTR 3.06× expected. **Confidence: medium-high.** Wrong if high CTR is explained by query mix or brand intent rather than content quality.

10. **Rank 10 — REVIEW.** 22,407 impressions, +75 clicks, and CTR 1.97× expected. **Confidence: medium.** Wrong if the observed increase reflects a stable upward trend.

11. **Rank 11 — REVIEW.** 21,540 impressions, +45 clicks, CTR 2.34× expected, with average position near 10. **Confidence: medium.** Wrong if ranking improvements explain the change without any need for content work.

12. **Rank 12 — REVIEW.** 21,459 impressions, +28 clicks, and CTR 2.91× expected. **Confidence: medium.** Wrong if the small absolute click change is too weak to represent a meaningful signal.

13. **Rank 13 — REVIEW.** 20,424 impressions, +67 clicks, and CTR 1.93× expected. **Confidence: medium.** Wrong if recent demand remains stable after the decision date.

14. **Rank 14 — REVIEW.** 19,716 impressions and +49 clicks, but CTR is only 1.31× expected. **Confidence: low-medium.** Wrong if this item crossed the threshold only because of ordinary measurement variation.

15. **Rank 15 — REVIEW.** 19,624 impressions, +24 clicks, and CTR 1.71× expected. **Confidence: medium-low.** Wrong if the small click increase does not represent a real change in user demand.

16. **Rank 16 — REVIEW.** 19,503 impressions and +19 clicks, with CTR only 1.20× expected. **Confidence: low.** This is one of the weakest picks because it barely meets the CTR rule. It would be wrong if a small change in the data moved it below the threshold.

17. **Rank 17 — REVIEW.** 19,063 impressions, +60 clicks, and CTR 2.95× expected. **Confidence: medium-high.** Wrong if the strong CTR represents healthy persistent performance rather than a temporary surge.

18. **Rank 18 — REVIEW.** 18,590 impressions, +100 clicks, and CTR 2.67× expected. **Confidence: medium-high.** Wrong if the click increase is explained by temporary search demand rather than something requiring editorial attention.

19. **Rank 19 — REVIEW.** 17,768 impressions, +53 clicks, and CTR 1.88× expected. **Confidence: medium.** Wrong if current momentum continues and there is no subsequent deterioration.

20. **Rank 20 — REVIEW.** 17,651 impressions, +67 clicks, and CTR 1.83× expected. **Confidence: medium.** Wrong if the recent increase reflects a lasting improvement rather than short-term variation.

In [5]:
# Section 3 — Top-20 review table
# No client/content identifiers are displayed.

top20_review = queue.head(20)[
    [
        "rank",
        "action_label",
        "reason_code",
        "recent_impressions_7d",
        "recent_clicks_7d",
        "recent_avg_position",
        "recent_ctr",
        "ctr_vs_expected",
        "pre_decision_click_change_7d",
    ]
].copy()

# Make CTR easier to read
top20_review["recent_ctr_percent"] = (
    top20_review["recent_ctr"] * 100
).round(3)

top20_review["ctr_vs_expected"] = (
    top20_review["ctr_vs_expected"]
).round(2)

top20_review["recent_avg_position"] = (
    top20_review["recent_avg_position"]
).round(2)

top20_review = top20_review.drop(
    columns=["recent_ctr"]
)

print("Top-20 review inputs — pseudonymized IDs are intentionally hidden.")
display(top20_review)

print(
    "\nObserved Precision@20: "
    f"{precision_at_20:.2%}"
)

print(
    "Observed base rate: "
    f"{base_rate:.2%}"
)

print(
    "The top-20 result does not show lift over the base rate, "
    "so these recommendations should be treated as low-confidence "
    "decision-support rather than automatic refresh actions."
)


Top-20 review inputs — pseudonymized IDs are intentionally hidden.


,rank,action_label,reason_code,recent_impressions_7d,recent_clicks_7d,recent_avg_position,ctr_vs_expected,pre_decision_click_change_7d,recent_ctr_percent
0,1,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,52735.0,710.0,4.24,3.37,219.0,1.346
1,2,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,41251.0,624.0,3.14,3.78,204.0,1.513
2,3,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,34792.0,227.0,11.97,1.41,158.0,0.652
3,4,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,33994.0,167.0,3.77,1.23,98.0,0.491
4,5,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,30538.0,201.0,5.25,1.65,32.0,0.658
5,6,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,29010.0,280.0,3.51,2.41,186.0,0.965
6,7,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,28999.0,170.0,3.37,1.47,66.0,0.586
7,8,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,24453.0,215.0,2.78,2.39,121.0,0.879
8,9,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,22465.0,275.0,3.37,3.06,137.0,1.224
9,10,REVIEW,RECENT_SURGE_ABOVE_EXPECTED_CTR,22407.0,177.0,3.05,1.97,75.0,0.790



Observed Precision@20: 55.00%
Observed base rate: 55.04%
The top-20 result does not show lift over the base rate, so these recommendations should be treated as low-confidence decision-support rather than automatic refresh actions.


## 4. Weak picks + leakage check
### Weak picks and leakage check

The baseline did not beat the base rate at the top 20, so the ranked queue should be treated as a weak but transparent benchmark.

The weakest-looking top-20 picks are ranks 4, 14, and 16. Their CTR ratios are close to the 1.20 eligibility threshold, so small changes in the data could remove them from the queue. Rank 16 is especially fragile at approximately 1.20× expected CTR.

This review also shows a limitation of ranking eligible rows only by impressions. High exposure makes an item important to inspect, but exposure alone does not mean the rule is good at predicting the future outcome.

No future-window or label-derived value is used to create `baseline_action_score`. Future clicks and `future_click_decline` are retained only for retrospective evaluation. Client/content hash IDs are grouping keys and are not scoring inputs.

I will not tune the rule after seeing the top-20 outcome because doing so on the same evaluation window would make the baseline less honest.

In [6]:
# Section 4 — Weak picks and leakage assertions

score_inputs = [
    "recent_impressions_7d",
    "pre_decision_click_change_7d",
    "ctr_vs_expected",
]

forbidden_score_inputs = [
    "future_clicks_7d",
    "future_click_decline",
    "is_future_click_decline",
    "label_copy_leak",
]

print("Inputs used by the baseline rule:")
for col in score_inputs:
    print("-", col)

# Confirm none of the forbidden future/label fields are rule inputs
assert not any(
    col in score_inputs
    for col in forbidden_score_inputs
)

# Confirm required queue output does not contain future outcome fields
assert "future_clicks_7d" not in queue_output.columns
assert "future_click_decline" not in queue_output.columns

weak_picks = queue.head(20).nsmallest(
    3,
    "ctr_vs_expected"
)[
    [
        "rank",
        "baseline_action_score",
        "recent_impressions_7d",
        "pre_decision_click_change_7d",
        "ctr_vs_expected",
    ]
].copy()

weak_picks["ctr_vs_expected"] = (
    weak_picks["ctr_vs_expected"]
).round(3)

print("\nThree threshold-fragile picks in the Top 20:")
display(weak_picks)

print(
    "\nPASS: No future-window or label-derived fields "
    "are used in the baseline score."
)

print(
    "PASS: The CSV queue contains decision-time fields only."
)

print(
    f"\nBaseline Precision@20 = {precision_at_20:.4f}"
)
print(
    f"Base rate = {base_rate:.4f}"
)

lift = precision_at_20 - base_rate

print(
    f"Absolute lift over base rate = {lift:+.4f}"
)


Inputs used by the baseline rule:
- recent_impressions_7d
- pre_decision_click_change_7d
- ctr_vs_expected

Three threshold-fragile picks in the Top 20:


,rank,baseline_action_score,recent_impressions_7d,pre_decision_click_change_7d,ctr_vs_expected
15,16,19503.0,19503.0,19.0,1.205
3,4,33994.0,33994.0,98.0,1.228
13,14,19716.0,19716.0,49.0,1.306



PASS: No future-window or label-derived fields are used in the baseline score.
PASS: The CSV queue contains decision-time fields only.

Baseline Precision@20 = 0.5500
Base rate = 0.5504
Absolute lift over base rate = -0.0004


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.